# Echo-TOF WIFF2 Data Processing & Compound Verification

WIFF2 파일 데이터를 처리하고, AI 추론으로 부반응 생성물을 예측하여 검증하는 파이프라인입니다.

| Step | 내용 |
|------|------|
| 1 | 환경 설정 및 라이브러리 임포트 |
| 2 | WIFF2 파일 로드 및 초기 데이터 가시화 (Custom Data 생성) |
| 3 | 적분 로직 적용 및 결과 가시화 |
| 4 | 화합물 구조 입력 및 패턴 가시화 |
| 5 | AI 추론 — 부반응 생성물(Side Products) 예측 & 탐색 |
| 6 | 3중 검증 (Mass Accuracy + Isotope Pattern + Adduct Consistency) |
| 7 | 최종 판정 & 리포트 |


## Step 1. 환경 설정 및 라이브러리 임포트

WIFF2 파일 처리, 데이터 분석, 시각화에 필요한 라이브러리를 임포트합니다.


In [ ]:
# ─── 패키지 설치 (Colab 환경) ───
!pip install -q numpy scipy matplotlib seaborn pandas
# WIFF2/mzML 처리 시:
# !pip install -q pyopenms
# 구조 시각화 시:
# !pip install -q rdkit-pypi


In [ ]:
# ─── echo_tof 패키지 로드 ───
# Colab: GitHub에서 clone
!git clone -q https://github.com/chsong-hash/echo-tof-colab.git /content/etof 2>/dev/null || true
import sys, os
sys.path.insert(0, '/content/etof/src')
# 로컬 실행 시: sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

# ─── echo_tof core ───
from echo_tof.elements import parse_formula, get_monoisotopic_mass
from echo_tof.molecule import Molecule
from echo_tof.isotope_calc import IsotopicDistributionCalculator
from echo_tof.pattern import MoleculePattern
from echo_tof.filters import FormulaFilter
from echo_tof.calculations import get_mass_error
from echo_tof.pipeline import FormulaFinderPipeline

# ─── echo_tof_ext ───
from echo_tof_ext.di_spectrum import pick_peaks, group_isotope_clusters, extract_cluster_at_mz
from echo_tof_ext.isotope_verifier import IsotopeVerifier
from echo_tof_ext.mz_predict import predict_mz, ADDUCTS
from echo_tof_ext.reaction_predictor import predict_product_mw, predict_byproduct_mws, validate_compound_by_adducts
from echo_tof_ext.peak_classifier import PeakClassifier, PeakClassification
from echo_tof_ext.neutral_loss_db import NEUTRAL_LOSSES, DELTA_MZ_PATTERNS

# ─── pyOpenMS (선택) ───
try:
    import pyopenms as oms
    HAS_OPENMS = True
    print(f"pyOpenMS {oms.__version__} loaded")
except ImportError:
    HAS_OPENMS = False
    print("pyOpenMS not available (mzML 로드 시 필요)")

# ─── RDKit (선택) ───
try:
    from rdkit import Chem
    from rdkit.Chem import Draw, Descriptors
    HAS_RDKIT = True
    print("RDKit loaded")
except ImportError:
    HAS_RDKIT = False
    print("RDKit not available (구조 시각화 시 필요)")

print("\nAll modules ready!")


## Step 2. WIFF2 파일 로드 및 초기 데이터 가시화

### 2.1 데이터 로드

WIFF2는 SCIEX 독점 포맷이므로 ProteoWizard `msconvert`로 mzML 변환 후 사용을 권장합니다.
또는 SCIEX OS에서 텍스트/CSV로 내보낸 데이터를 사용할 수 있습니다.

> **데모 모드**: 실제 파일 없이도 시뮬레이션 데이터로 전체 워크플로우를 실행할 수 있습니다.


In [ ]:
# ─── 데이터 소스 설정 ───
# 옵션 1: Google Drive에서 mzML 로드
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/EchoTOF_Data/sample.mzML'

# 옵션 2: 파일 업로드
# from google.colab import files
# uploaded = files.upload()

# 옵션 3: 데모 모드 (시뮬레이션)
USE_DEMO = True
print("Demo mode: ON" if USE_DEMO else "Loading real data...")


In [ ]:
# ─── mzML 로드 함수 (실제 데이터용) ───
def load_mzml(filepath):
    \"\"\"mzML 파일 → 스펙트럼 리스트 + TIC + 메타데이터\"\"\"
    if not HAS_OPENMS:
        raise ImportError("pip install pyopenms")
    exp = oms.MSExperiment()
    oms.MzMLFile().load(str(filepath), exp)
    spectra = []
    for s in exp:
        mz_arr, int_arr = s.get_peaks()
        spectra.append({'rt': s.getRT(), 'ms_level': s.getMSLevel(),
                        'mz': mz_arr, 'intensity': int_arr,
                        'tic': np.sum(int_arr)})
    print(f"Loaded {len(spectra)} spectra, RT: {spectra[0]['rt']:.1f}-{spectra[-1]['rt']:.1f} sec")
    return spectra

def extract_tic(spectra):
    \"\"\"전체 이온 크로마토그램 (TIC)\"\"\"
    ms1 = [s for s in spectra if s['ms_level'] == 1]
    return np.array([s['rt'] for s in ms1]), np.array([s['tic'] for s in ms1])

def extract_eic(spectra, target_mz, tol_da=0.01):
    \"\"\"추출 이온 크로마토그램 (XIC/EIC)\"\"\"
    rts, ints = [], []
    for s in spectra:
        if s['ms_level'] != 1: continue
        mask = np.abs(s['mz'] - target_mz) <= tol_da
        rts.append(s['rt'])
        ints.append(np.sum(s['intensity'][mask]) if mask.any() else 0)
    return np.array(rts), np.array(ints)

print("mzML loader functions defined.")
print("  load_mzml(path) -> spectra list")
print("  extract_tic(spectra) -> (rt, tic)")
print("  extract_eic(spectra, target_mz) -> (rt, eic)")


### 2.2 Custom Data 생성 & 초기 가시화

데모 모드에서는 실제 의약합성 반응 crude 스펙트럼을 시뮬레이션합니다.

**시나리오**: Amide coupling 반응
- **SM** (Starting Material): Aniline 유도체, MW = 197.084
- **Product**: Amide 생성물, MW = 315.125
- **Side products**: 미반응 SM, 이합체, 탈수체 등


In [ ]:
# ─── Custom Data 생성 (반응 crude 시뮬레이션) ───
SM_FORMULA = "C12H11NO2"
SM_MW = 201.079
PRODUCT_FORMULA = "C18H18N2O3"
PRODUCT_MW = 310.131
REAGENT_MW = 127.063  # benzoic acid
REACTION_TYPE = "amide_coupling"

PROTON = 1.00728

np.random.seed(42)
mz = np.linspace(50, 600, 8000)
intensity = np.zeros_like(mz)

def add_peak(center, height, width=0.015):
    global intensity
    intensity += height * np.exp(-((mz - center)**2) / (2 * width**2))

# ── SM 피크 (미반응, 약간 남음) ──
add_peak(SM_MW + PROTON, 3000)                    # [M+H]+
add_peak(SM_MW + PROTON + 1.003, 210)             # M+1
add_peak(SM_MW + 22.989, 500)                     # [M+Na]+

# ── Product 피크 (주 생성물) ──
add_peak(PRODUCT_MW + PROTON, 10000)              # [M+H]+
add_peak(PRODUCT_MW + PROTON + 1.003, 900)        # M+1
add_peak(PRODUCT_MW + PROTON + 2.006, 50)         # M+2
add_peak(PRODUCT_MW + 22.989, 1800)               # [M+Na]+
add_peak(PRODUCT_MW + 38.963, 300)                # [M+K]+

# ── Side products (AI가 추론해야 할 것들) ──
# 탈수체 (-H2O)
add_peak(PRODUCT_MW + PROTON - 18.011, 800)       # [Product-H2O+H]+
# SM 이합체
add_peak(2 * SM_MW + PROTON, 400)                 # [2SM+H]+
# 시약 잔류
add_peak(REAGENT_MW + PROTON, 600)                # Reagent [M+H]+
# 미지 불순물
add_peak(267.134, 350)                            # Unknown 1
add_peak(189.079, 250)                            # Unknown 2

# ── 프래그먼트 ──
add_peak(138.055, 1500)                           # product fragment
add_peak(110.060, 700)                            # product fragment

# 노이즈
intensity += np.abs(np.random.normal(0, 40, len(mz)))

# ─── Custom Data → DataFrame ───
custom_data = pd.DataFrame({'mz': mz, 'intensity': intensity})
print(f"Custom Data: {len(custom_data)} points")
print(f"m/z range: [{mz[0]:.0f}, {mz[-1]:.0f}]")
print(f"Max intensity: {intensity.max():.0f}")
custom_data.head()


In [ ]:
# ─── 초기 가시화: TIC 대응 + 질량 스펙트럼 ───
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# 전체 스펙트럼
ax = axes[0]
ax.plot(mz, intensity, color='#0078D4', lw=0.5)
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title('Mass Spectrum (Full Range)')

# SM 영역 (m/z 195-230)
ax = axes[1]
m = (mz >= 195) & (mz <= 230)
ax.plot(mz[m], intensity[m], color='#DC143C', lw=0.8)
ax.set_xlabel('m/z'); ax.set_title(f'SM Region (MW={SM_MW:.3f})')

# Product 영역 (m/z 305-355)
ax = axes[2]
m = (mz >= 305) & (mz <= 355)
ax.plot(mz[m], intensity[m], color='#2E8B57', lw=0.8)
ax.set_xlabel('m/z'); ax.set_title(f'Product Region (MW={PRODUCT_MW:.3f})')

plt.tight_layout(); plt.show()


## Step 3. 적분 로직 적용 및 결과 가시화

### 3.1 사용자 정의 적분 로직

> **VSCode에서 사용하던 로직을 여기에 삽입하세요.**
> 아래 `PeakIntegrator` 클래스의 `integrate()` 메서드를 수정하거나,
> 기존 코드를 `pick_peaks()` 호출 후에 추가하면 됩니다.


In [ ]:
# ─── 피크 검출 & 적분 (echo_tof_ext.di_spectrum) ───
# pick_peaks: MAD 노이즈 추정 → S/N 필터 → 로컬 최댓값
detected = pick_peaks(mz, intensity, noise_factor=3.0, min_intensity_pct=0.1)
peaks_df = pd.DataFrame(detected)

# 동위원소 클러스터링
clusters = group_isotope_clusters(detected, charge=1, mz_tolerance=0.02)

print(f"Detected peaks: {len(peaks_df)}")
print(f"Isotope clusters: {len(clusters)}")

# ────────────────────────────────────────────
# [USER] 여기에 기존 VSCode 적분 로직을 삽입하세요.
# 예시:
# from my_custom_module import custom_integrate
# peaks_df['custom_area'] = custom_integrate(mz, intensity, peaks_df)
# ────────────────────────────────────────────

peaks_df[['mz', 'intensity', 'area', 'sn']].round(2)


In [ ]:
# ─── 적분 결과 가시화 (피크 영역 표시) ───
fig, axes = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.plot(mz, intensity, color='#888', lw=0.4, alpha=0.5, label='Spectrum')
for _, pk in peaks_df.iterrows():
    idx = int(pk['index'])
    left = max(0, idx - 5)
    right = min(len(mz)-1, idx + 5)
    ax.fill_between(mz[left:right+1], 0, intensity[left:right+1],
                    alpha=0.3, color='#2196F3')
    if pk['intensity'] > np.max(intensity) * 0.03:
        ax.annotate(f"{pk['mz']:.3f}\nArea={pk['area']:.0f}",
            xy=(pk['mz'], pk['intensity']), xytext=(0, 12),
            textcoords='offset points', ha='center', fontsize=6, color='red')
ax.set_ylabel('Intensity')
ax.set_title(f'Peak Integration: {len(peaks_df)} peaks detected')
ax.legend(fontsize=8)

# 피크 면적 바 차트
ax = axes[1]
ax.bar(peaks_df['mz'], peaks_df['area'], width=1.5, color='#0078D4', alpha=0.7)
ax.set_xlabel('m/z'); ax.set_ylabel('Peak Area')
ax.set_title('Integrated Peak Areas')

plt.tight_layout(); plt.show()

# 적분 값 테이블
print(f"\nTop 10 peaks by area:")
peaks_df.nlargest(10, 'area')[['mz', 'intensity', 'area', 'sn']].round(2)


## Step 4. 화합물 구조 입력 및 패턴 가시화

### 4.1 화합물 정보 입력

SMILES, InChIKey, 또는 분자식으로 화합물을 정의합니다.
CSV에서 화합물 목록을 로드할 수도 있습니다.


In [ ]:
# ─── 화합물 정의 ───
# 방법 1: 직접 입력
COMPOUNDS = {
    "SM":      {"formula": SM_FORMULA, "smiles": "O=C(Oc1ccccc1)c1ccccc1N", "role": "SM"},
    "Product": {"formula": PRODUCT_FORMULA, "smiles": "O=C(Nc1ccccc1C(=O)Oc1ccccc1)c1ccccc1", "role": "Product"},
    "Reagent": {"formula": "C7H5NO", "role": "Reagent"},
}

# 방법 2: CSV에서 로드
# compound_df = pd.read_csv('compounds.csv')  # 컬럼: name, formula, smiles, role
# COMPOUNDS = {row['name']: row.to_dict() for _, row in compound_df.iterrows()}

# ─── MW 및 adduct m/z 계산 ───
for name, info in COMPOUNDS.items():
    mol = Molecule(info["formula"], charge=0)
    info["mw"] = mol.monoisotopic_mass
    info["adducts"] = predict_mz(info["mw"], mode="positive")
    info["mz_mh"] = info["mw"] + PROTON  # [M+H]+
    info["rdb"] = mol.rdb

# 테이블 출력
rows = []
for name, info in COMPOUNDS.items():
    rows.append({"Name": name, "Formula": info["formula"], "MW": f"{info['mw']:.4f}",
                 "[M+H]+": f"{info['mz_mh']:.4f}", "RDB": f"{info.get('rdb',0):.1f}",
                 "Role": info["role"]})
pd.DataFrame(rows)


In [ ]:
# ─── 화합물 구조 시각화 (RDKit) ───
if HAS_RDKIT:
    mols, legends = [], []
    for name, info in COMPOUNDS.items():
        smiles = info.get("smiles")
        if smiles:
            m = Chem.MolFromSmiles(smiles)
            if m:
                mols.append(m)
                legends.append(f"{name}\nMW={info['mw']:.3f}")
    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(350, 250), legends=legends)
        display(img)
else:
    print("RDKit not installed. 구조 시각화 생략.")
    for name, info in COMPOUNDS.items():
        print(f"  {name}: {info['formula']} (MW={info['mw']:.4f})")


In [ ]:
# ─── 화합물-피크 매핑 & 패턴 가시화 ───
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(mz, intensity, color='#ccc', lw=0.5)

ROLE_COLOR = {"SM": "#DC143C", "Product": "#2E8B57", "Reagent": "#FF8C00"}

for name, info in COMPOUNDS.items():
    color = ROLE_COLOR.get(info["role"], "#999")
    for a in info["adducts"][:3]:  # top 3 adducts
        idx = np.argmin(np.abs(mz - a["mz"]))
        if intensity[idx] > np.median(intensity) * 3:
            ax.fill_between(mz[max(0,idx-4):idx+5], 0, intensity[max(0,idx-4):idx+5],
                          color=color, alpha=0.4)
            ax.annotate(f"{name}\n{a['adduct']}\n{a['mz']:.3f}",
                xy=(a['mz'], intensity[idx]), xytext=(0, 15),
                textcoords='offset points', ha='center', fontsize=7,
                color=color, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=r) for r, c in ROLE_COLOR.items()], fontsize=9)
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title('Step 4: Compound-Peak Assignment')
plt.tight_layout(); plt.show()


## Step 5. AI 추론 — 부반응 생성물(Side Products) 예측 & 탐색

반응 유형과 출발물질 정보로부터:
1. **예상 부산물 MW 자동 예측** (`predict_byproduct_mws`)
2. 각 부산물의 **adduct m/z 생성** (`predict_mz`)
3. 중성 손실 패턴 DB로 **추가 side product 추론** (`NEUTRAL_LOSSES`, `DELTA_MZ_PATTERNS`)
4. 스펙트럼에서 **자동 탐색** + 검출 여부 판정


In [ ]:
# ─── 5.1 반응 기반 부산물 예측 ───
print("=" * 60)
print("  AI Inference: Side Product Prediction")
print("=" * 60)

# 반응 유형 기반 예측
byproducts = predict_byproduct_mws(SM_MW, REACTION_TYPE, REAGENT_MW)
print(f"\nReaction type: {REACTION_TYPE}")
print(f"Predicted byproducts: {len(byproducts)}")

# 각 부산물의 adduct m/z 계산
for bp in byproducts:
    bp["adducts"] = predict_mz(bp["mw"], mode="positive")
    bp["mz_mh"] = bp["mw"] + PROTON
    print(f"  {bp['name']:35s} MW={bp['mw']:.3f}  [M+H]+={bp['mz_mh']:.4f}  ({bp['likelihood']})")

# ─── 중성 손실 기반 추가 추론 ───
print(f"\nNeutral loss patterns: {len(DELTA_MZ_PATTERNS)} types")
product_mh = PRODUCT_MW + PROTON
nl_candidates = []
for nl_name, nl_mass, _ in NEUTRAL_LOSSES[:20]:  # 상위 20개
    expected_mz = product_mh - nl_mass
    if 50 < expected_mz < 600:
        nl_candidates.append({"name": f"Product - {nl_name}", "mz": expected_mz, "loss": nl_mass, "type": "neutral_loss"})

print(f"Neutral loss candidates from Product: {len(nl_candidates)}")
for nl in nl_candidates[:5]:
    print(f"  {nl['name']:35s} m/z={nl['mz']:.4f}  (loss={nl['loss']:.4f})")
print(f"  ... and {len(nl_candidates)-5} more")


In [ ]:
# ─── 5.2 스펙트럼 탐색: 예측된 side product가 있는가? ───
MZ_TOL = 0.03  # Da

peak_mzs = np.array([p['mz'] for p in detected])
peak_ints = np.array([p['intensity'] for p in detected])

# 부산물 탐색
print("=" * 60)
print("  Side Product Detection Results")
print("=" * 60)

all_side_products = []

# (A) 반응 예측 부산물
for bp in byproducts:
    found = False
    for a in bp["adducts"][:3]:
        matches = np.where(np.abs(peak_mzs - a["mz"]) < MZ_TOL)[0]
        if len(matches) > 0:
            best = matches[np.argmax(peak_ints[matches])]
            err_ppm = (peak_mzs[best] - a["mz"]) / a["mz"] * 1e6
            all_side_products.append({
                "name": bp["name"], "type": "reaction_byproduct",
                "expected_mz": a["mz"], "observed_mz": peak_mzs[best],
                "adduct": a["adduct"], "intensity": peak_ints[best],
                "error_ppm": err_ppm, "detected": True,
                "likelihood": bp["likelihood"],
            })
            found = True
            break
    if not found:
        all_side_products.append({
            "name": bp["name"], "type": "reaction_byproduct",
            "expected_mz": bp["mz_mh"], "observed_mz": 0,
            "adduct": "[M+H]+", "intensity": 0,
            "error_ppm": 0, "detected": False,
            "likelihood": bp["likelihood"],
        })

# (B) 중성 손실 탐색
for nl in nl_candidates:
    matches = np.where(np.abs(peak_mzs - nl["mz"]) < MZ_TOL)[0]
    if len(matches) > 0:
        best = matches[np.argmax(peak_ints[matches])]
        err_ppm = (peak_mzs[best] - nl["mz"]) / nl["mz"] * 1e6
        all_side_products.append({
            "name": nl["name"], "type": "neutral_loss",
            "expected_mz": nl["mz"], "observed_mz": peak_mzs[best],
            "adduct": "NL", "intensity": peak_ints[best],
            "error_ppm": err_ppm, "detected": True,
            "likelihood": "inferred",
        })

# 결과 출력
det = [s for s in all_side_products if s["detected"]]
undet = [s for s in all_side_products if not s["detected"]]
print(f"\nDetected: {len(det)}")
for s in det:
    print(f"  [FOUND]   {s['name']:35s} expected={s['expected_mz']:.4f}  "
          f"observed={s['observed_mz']:.4f}  err={s['error_ppm']:.1f}ppm  "
          f"int={s['intensity']:.0f}")
print(f"\nNot detected: {len(undet)}")
for s in undet:
    print(f"  [ABSENT]  {s['name']:35s} expected={s['expected_mz']:.4f}")


In [ ]:
# ─── 5.3 AI 추론 결과 가시화 ───
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(mz, intensity, color='#ddd', lw=0.5)

# Known compounds (Step 4)
for name, info in COMPOUNDS.items():
    color = ROLE_COLOR.get(info["role"], "#999")
    idx = np.argmin(np.abs(mz - info["mz_mh"]))
    ax.fill_between(mz[max(0,idx-4):idx+5], 0, intensity[max(0,idx-4):idx+5],
                  color=color, alpha=0.3)

# Side products (Step 5)
for s in all_side_products:
    if not s["detected"]: continue
    idx = np.argmin(np.abs(mz - s["observed_mz"]))
    color = '#FF6B35' if s["type"] == "reaction_byproduct" else '#9B59B6'
    ax.fill_between(mz[max(0,idx-4):idx+5], 0, intensity[max(0,idx-4):idx+5],
                  color=color, alpha=0.4)
    ax.annotate(f"{s['name']}\n{s['observed_mz']:.3f}",
        xy=(s['observed_mz'], s['intensity']), xytext=(0, 12),
        textcoords='offset points', ha='center', fontsize=6,
        color=color, fontweight='bold')

# Unknown (나머지)
known_mzs = set()
for info in COMPOUNDS.values():
    for a in info["adducts"][:3]: known_mzs.add(round(a["mz"], 2))
for s in all_side_products:
    if s["detected"]: known_mzs.add(round(s["observed_mz"], 2))

for _, pk in peaks_df.iterrows():
    if not any(abs(pk['mz'] - km) < 0.1 for km in known_mzs):
        if pk['intensity'] > np.max(intensity) * 0.02:
            idx = int(pk['index'])
            ax.fill_between(mz[max(0,idx-3):idx+4], 0, intensity[max(0,idx-3):idx+4],
                          color='#999', alpha=0.3)
            ax.annotate(f"Unknown\n{pk['mz']:.3f}", xy=(pk['mz'], pk['intensity']),
                xytext=(0, 8), textcoords='offset points', ha='center',
                fontsize=6, color='#999')

ax.legend(handles=[
    Patch(color='#2E8B57', label='Product'), Patch(color='#DC143C', label='SM'),
    Patch(color='#FF6B35', label='Reaction Byproduct'), Patch(color='#9B59B6', label='Neutral Loss'),
    Patch(color='#999', label='Unknown'),
], fontsize=8, loc='upper right')
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title('Step 5: AI Inference — Known + Side Products + Unknown')
plt.tight_layout(); plt.show()


## Step 6. 3중 검증

검출된 각 화합물에 대해 **3가지 검증**을 수행합니다:

| 검증 | 방법 | 기준 |
|------|------|------|
| 6a. Mass Accuracy | `get_mass_error()` | < 5 ppm |
| 6b. Isotope Pattern | `IsotopeVerifier.verify()` | grade = excellent / good |
| 6c. Adduct Consistency | `validate_compound_by_adducts()` | 2개 이상 adduct 검출 |


In [ ]:
# ─── 6a + 6b + 6c 통합 검증 ───
verifier = IsotopeVerifier(ppm_tolerance=10.0)
idc = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)

verification_results = []

for name, info in COMPOUNDS.items():
    print(f"\n{'='*50}")
    print(f"  Verifying: {name} ({info['formula']})")
    print(f"{'='*50}")

    result = {"name": name, "formula": info["formula"], "role": info["role"]}

    # 6a. Mass Accuracy
    mh_mz = info["mz_mh"]
    matches = np.where(np.abs(peak_mzs - mh_mz) < MZ_TOL)[0]
    if len(matches) > 0:
        best_idx = matches[np.argmax(peak_ints[matches])]
        err = abs((peak_mzs[best_idx] - mh_mz) / mh_mz * 1e6)
        result["mass_ppm"] = err
        result["mass_pass"] = err < 5.0
        print(f"  6a Mass Accuracy: {err:.2f} ppm {'PASS' if err < 5 else 'FAIL'}")
    else:
        result["mass_ppm"] = 999
        result["mass_pass"] = False
        print(f"  6a Mass Accuracy: [M+H]+ NOT FOUND")

    # 6b. Isotope Pattern
    cluster = extract_cluster_at_mz(mz, intensity, mh_mz, charge=1, mz_tolerance_ppm=10.0, n_isotopes=5)
    if cluster["found"]:
        vr = verifier.verify(info["formula"], 1, cluster["mz_array"], cluster["int_array"])
        result["isotope_grade"] = vr["grade"]
        result["isotope_pass"] = vr["grade"] in ("excellent", "good")
        result["cluster_error"] = vr["cluster_error"]
        print(f"  6b Isotope Pattern: grade={vr['grade'].upper()}, CE={vr['cluster_error']:.4f} "
              f"{'PASS' if result['isotope_pass'] else 'FAIL'}")
    else:
        result["isotope_grade"] = "not_found"
        result["isotope_pass"] = False
        result["cluster_error"] = 999
        print(f"  6b Isotope Pattern: cluster NOT FOUND")

    # 6c. Adduct Consistency
    validation = validate_compound_by_adducts(peak_mzs, peak_ints, info["mw"], MZ_TOL, "positive")
    result["n_adducts"] = validation["n_found"]
    result["adduct_pass"] = validation["n_found"] >= 2
    print(f"  6c Adduct Consistency: {validation['n_found']}/{validation['n_possible']} adducts "
          f"{'PASS' if result['adduct_pass'] else 'FAIL'}")

    # 판정
    checks = [result["mass_pass"], result["isotope_pass"], result["adduct_pass"]]
    n = sum(checks)
    result["n_pass"] = n
    result["judgment"] = {3:"CONFIRMED", 2:"PROBABLE", 1:"TENTATIVE", 0:"NOT DETECTED"}[n]
    print(f"  >>> JUDGMENT: {result['judgment']} ({n}/3)")

    verification_results.append(result)


## Step 7. 최종 판정 & 리포트

In [ ]:
# ─── 종합 리포트 테이블 ───
report_rows = []
for r in verification_results:
    report_rows.append({
        "Compound": r["name"],
        "Formula": r["formula"],
        "Role": r["role"],
        "Mass (ppm)": f"{r['mass_ppm']:.1f}" if r['mass_ppm'] < 900 else "N/A",
        "Isotope": r.get("isotope_grade", "N/A").upper(),
        "Adducts": f"{r['n_adducts']}",
        "Judgment": r["judgment"],
        "Score": f"{r['n_pass']}/3",
    })
report_df = pd.DataFrame(report_rows)

# Side products 요약 추가
side_det = [s for s in all_side_products if s["detected"]]
print("=" * 70)
print("  COMPOUND VERIFICATION REPORT")
print("=" * 70)
print(f"\nKnown compounds: {len(verification_results)}")
print(f"Side products detected: {len(side_det)}")
print(f"Total peaks explained: {len(det) + sum(1 for r in verification_results if r['mass_pass'])}")
print(f"Unknown peaks: {len(peaks_df) - len(det) - sum(1 for r in verification_results if r['mass_pass'])}")
print()
report_df


In [ ]:
# ─── 종합 가시화 ───
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 왼쪽: 검증 결과 스펙트럼
JCOLOR = {"CONFIRMED":"#2E8B57", "PROBABLE":"#FF8C00", "TENTATIVE":"#DC143C", "NOT DETECTED":"#999"}
ax = axes[0]
ax.plot(mz, intensity, color='#ddd', lw=0.5)
for r in verification_results:
    info = COMPOUNDS[r["name"]]
    color = JCOLOR[r["judgment"]]
    for a in info["adducts"][:2]:
        idx = np.argmin(np.abs(mz - a["mz"]))
        ax.fill_between(mz[max(0,idx-4):idx+5], 0, intensity[max(0,idx-4):idx+5],
                      color=color, alpha=0.4)
    ax.annotate(f"{r['name']}\n{r['judgment']}", xy=(info['mz_mh'], intensity[np.argmin(np.abs(mz-info['mz_mh']))]),
        xytext=(0, 15), textcoords='offset points', ha='center', fontsize=8,
        color=color, fontweight='bold')
ax.legend(handles=[Patch(color=c, label=j) for j, c in JCOLOR.items()], fontsize=8)
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title('Verification Results')

# 오른쪽: 검증 매트릭스
ax = axes[1]
names = [r["name"] for r in verification_results]
checks_labels = ["Mass\nAccuracy", "Isotope\nPattern", "Adduct\nConsist."]
matrix = np.array([[r["mass_pass"], r["isotope_pass"], r["adduct_pass"]] for r in verification_results], dtype=int)
cmap = plt.cm.colors.ListedColormap(['#DC143C', '#2E8B57'])
ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(checks_labels, fontsize=9)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=10)
for i in range(len(names)):
    for j in range(3):
        ax.text(j, i, "PASS" if matrix[i,j] else "FAIL", ha='center', va='center',
               fontsize=10, fontweight='bold', color='white')
ax.set_title('Verification Matrix')
plt.tight_layout(); plt.show()


---
## 다음 단계

1. **실제 WIFF2 데이터**: `msconvert sample.wiff2 --mzML` → `load_mzml()` 사용
2. **VSCode 적분 로직**: Step 3 placeholder에 기존 코드 삽입
3. **다중 화합물**: `COMPOUNDS` 딕셔너리에 추가
4. **96웰 배치**: for loop으로 웰별 반복 실행 → 히트맵
5. **MS/MS 프래그먼트**: 4번째 검증 레이어 추가 가능
